# ---------- Weekdays vs weekends analysis notebook ----------

The goal of this notebook is to build two networks: one for weekdays and one for weekends. Then, the stationary distribution is computed for each network in order to identify the most important stations.

Author: Artur Werys

In [ ]:
from pathlib import Path

import math

import pandas as pd
import numpy as np
import seaborn as sns


import networkx as nx

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch

import geopandas as gpd
import contextily as ctx
import osmnx as ox

In [ ]:
def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start)
    for path in [current, *current.parents]:
        if (path / "Data").exists():
            return path
    raise FileNotFoundError("Could not find project root containing Data/")


PROJECT_ROOT = find_project_root()
BASE_DIR = PROJECT_ROOT
DATA_DIR = PROJECT_ROOT / "Data"

DATA_FILE = DATA_DIR / "final_trip_data.parquet"
PAIR_ROAD_DISTANCES_FILE = DATA_DIR / "station_pair_road_distances.parquet"

SCALE_TICK_INTERVAL_KM = 0.5

padding = 2800


In [ ]:
trip_data = pd.read_parquet(DATA_FILE)
trip_data.head()

In [ ]:
# start_day_of_week is prepared in data_preparation.ipynb.
trip_data.head()


In [ ]:
trip_data_weekends = trip_data[trip_data["start_day_of_week"].isin(["Saturday", "Sunday"])]
trip_data_weekdays = trip_data[trip_data["start_day_of_week"].isin(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"])]

station_pair_road_distances = pd.read_parquet(PAIR_ROAD_DISTANCES_FILE)


def route_lengths_for_day_type(day_trip_data, pair_road_distances):

    trip_counts = (
        day_trip_data
        .groupby(["start_station_name", "end_station_name"])
        .size()
        .rename("trip_count")
        .reset_index()
    )

    trip_counts = trip_counts[
        trip_counts["start_station_name"] != trip_counts["end_station_name"]
    ].copy()

    route_lengths = trip_counts.merge(
        pair_road_distances[
            ["start_station_name", "end_station_name", "road_distance_km"]
        ],
        on=["start_station_name", "end_station_name"],
        how="inner"
    )

    return route_lengths


def weighted_average_route_km(route_lengths):

    if route_lengths.empty:
        return np.nan

    return np.average(
        route_lengths["road_distance_km"],
        weights=route_lengths["trip_count"]
    )


def route_length_summary_row(label, day_trip_data, route_lengths):

    non_loop_trips = int((
        day_trip_data["start_station_name"] != day_trip_data["end_station_name"]
    ).sum())

    covered_trips = int(route_lengths["trip_count"].sum())

    return {
        "typ_dnia": label,
        "srednia_dlugosc_trasy_km": weighted_average_route_km(route_lengths),
        "liczba_przejazdow_z_odlegloscia": covered_trips,
        "pokrycie_przejazdow_%": covered_trips / non_loop_trips * 100 if non_loop_trips else np.nan,
    }


weekend_route_lengths = route_lengths_for_day_type(
    trip_data_weekends,
    station_pair_road_distances
)

weekday_route_lengths = route_lengths_for_day_type(
    trip_data_weekdays,
    station_pair_road_distances
)

average_weekend_route_km = weighted_average_route_km(weekend_route_lengths)
average_weekday_route_km = weighted_average_route_km(weekday_route_lengths)

average_route_length_summary = pd.DataFrame([
    route_length_summary_row("Weekendy", trip_data_weekends, weekend_route_lengths),
    route_length_summary_row("Dni robocze", trip_data_weekdays, weekday_route_lengths),
])

display(average_route_length_summary.round(2))


def mercator_latitude_from_y(y):

    web_mercator_radius_m = 6_378_137

    return math.degrees(
        2 * math.atan(math.exp(y / web_mercator_radius_m)) - math.pi / 2
    )


def web_mercator_width_for_ground_distance(ax, distance_km):

    y_min, y_max = ax.get_ylim()
    center_lat = mercator_latitude_from_y((y_min + y_max) / 2)
    latitude_scale = max(math.cos(math.radians(center_lat)), 0.1)

    return distance_km * 1000 / latitude_scale


def format_scale_distance_label(distance_km, include_unit=False):

    if include_unit:
        return f"{distance_km:.2f} km"

    rounded_distance = round(distance_km, 1)

    if math.isclose(rounded_distance, round(rounded_distance), abs_tol=0.01):
        return f"{round(rounded_distance):.0f}"

    return f"{rounded_distance:.1f}"


def scale_tick_marks(distance_km, tick_interval_km=SCALE_TICK_INTERVAL_KM):

    if distance_km <= 0:
        return [(0.0, 0.0)]

    marks = []
    tick_count = int(math.floor(distance_km / tick_interval_km))

    for tick_index in range(tick_count + 1):
        tick_distance_km = tick_index * tick_interval_km
        marks.append((tick_distance_km / distance_km, tick_distance_km))

    if not math.isclose(marks[-1][0], 1.0):
        marks.append((1.0, distance_km))

    return marks


def add_average_route_scale_bar(ax, average_route_km, label="Średnia trasa"):

    if average_route_km is None or not np.isfinite(average_route_km) or average_route_km <= 0:
        return

    x_min, x_max = ax.get_xlim()
    map_width = x_max - x_min

    bar_width_axes = web_mercator_width_for_ground_distance(
        ax,
        average_route_km
    ) / map_width

    if bar_width_axes > 0.42:
        return

    panel_width_axes = max(bar_width_axes + 0.035, 0.22)
    panel_height_axes = 0.073

    scale_ax = ax.inset_axes(
        [0.028, 0.050, panel_width_axes, panel_height_axes],
        zorder=30
    )
    scale_ax.set_axis_off()
    scale_ax.set_xlim(0, 1)
    scale_ax.set_ylim(0, 1)

    panel = FancyBboxPatch(
        (0, 0),
        1,
        1,
        transform=scale_ax.transAxes,
        boxstyle="round,pad=0.010,rounding_size=0.035",
        facecolor="white",
        edgecolor="#B4BCC2",
        linewidth=0.8,
        alpha=0.86,
        clip_on=False,
        zorder=0
    )
    scale_ax.add_patch(panel)

    bar_fraction = bar_width_axes / panel_width_axes
    bar_x = (1 - bar_fraction) / 2
    bar_y = 0.39
    bar_color = "#232A32"

    scale_ax.text(
        0.5,
        0.73,
        f"{label}: {average_route_km:.2f} km",
        ha="center",
        va="center",
        fontsize=8,
        fontweight="bold",
        color=bar_color,
        transform=scale_ax.transAxes,
        zorder=2
    )

    scale_ax.plot(
        [bar_x, bar_x + bar_fraction],
        [bar_y, bar_y],
        color=bar_color,
        linewidth=1.6,
        transform=scale_ax.transAxes,
        zorder=2
    )

    selected_labels = []
    min_label_gap = 0.11
    tick_marks = scale_tick_marks(average_route_km)

    for tick_index, (tick_fraction, tick_distance_km) in enumerate(tick_marks):

        tick_x = bar_x + bar_fraction * tick_fraction

        scale_ax.plot(
            [tick_x, tick_x],
            [bar_y - 0.09, bar_y + 0.09],
            color=bar_color,
            linewidth=0.9,
            transform=scale_ax.transAxes,
            zorder=2
        )

        is_final_tick = tick_index == len(tick_marks) - 1
        label_text = format_scale_distance_label(
            tick_distance_km,
            include_unit=is_final_tick
        )

        if is_final_tick:
            while selected_labels and tick_x - selected_labels[-1][0] < min_label_gap:
                selected_labels.pop()
            selected_labels.append((tick_x, label_text))

        elif not selected_labels or tick_x - selected_labels[-1][0] >= min_label_gap:
            selected_labels.append((tick_x, label_text))

    for label_x, label_text in selected_labels:

        scale_ax.text(
            label_x,
            0.14,
            label_text,
            ha="center",
            va="center",
            fontsize=7,
            color=bar_color,
            transform=scale_ax.transAxes,
            zorder=2
        )


def safe_add_basemap(ax, source=ctx.providers.CartoDB.Positron, alpha=0.58):

    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    try:
        ctx.add_basemap(
            ax,
            source=source
        )
    except Exception as error:
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
        print(
            "Nie udało się pobrać tła mapy z internetu. "
            "Rysuję stacje i podziałkę bez tła. "
            f"Szczegóły: {error.__class__.__name__}: {error}"
        )
        return False

    for image in ax.get_images():
        image.set_alpha(alpha)

    return True

In [ ]:
trip_data_weekends.head()

In [ ]:
trip_data_weekdays.head()

# Weekends network building

In [ ]:
start_stations_weekends_df = trip_data[
    [
        "start_station_id",
        "start_station_name",
        "start_lat",
        "start_lon",
    ]
].drop_duplicates()

start_stations_weekends_df = start_stations_weekends_df.rename(columns={
    "start_station_id": "station_id",
    "start_station_name": "station_name",
    "start_lat": "lat",
    "start_lon": "lon"
})

start_stations_weekends_df.head()

In [ ]:
end_stations_weekends_df = trip_data[
    [
        "end_station_id",
        "end_station_name",
        "end_lat",
        "end_lon",
    ]
].drop_duplicates()

end_stations_weekends_df = end_stations_weekends_df.rename(columns={
    "end_station_id": "station_id",
    "end_station_name": "station_name",
    "end_lat": "lat",
    "end_lon": "lon"
})

end_stations_weekends_df.head()

In [ ]:
stations_weekends_df = pd.concat([start_stations_weekends_df, end_stations_weekends_df], ignore_index=True).drop_duplicates()
print("Number of stations:", len(stations_weekends_df))

In [ ]:
edges_weekends_df = trip_data_weekends.groupby(
    [
        "start_station_id",
        "start_station_name",
        "end_station_id",
        "end_station_name"
    ]
).agg(
    weight=("start_station_id", "count")
).reset_index()

In [ ]:
graph_weekends = nx.from_pandas_edgelist(
    edges_weekends_df,
    source="start_station_name",
    target="end_station_name",
    edge_attr="weight",
    create_using=nx.DiGraph()
)

## Weekend Markov 

In [ ]:
positions = {}

for _, row in stations_weekends_df.iterrows():

    positions[row["station_name"]] = (
        row["lon"],
        row["lat"]
    )

station_to_index = {
    station_id: idx
    for idx, station_id in enumerate(
        sorted(stations_weekends_df["station_id"].unique())
    )
}

index_to_station = {
    idx: station_id
    for station_id, idx in station_to_index.items()
}

In [ ]:
N = len(stations_weekends_df["station_id"].unique())

wij_matrix = np.zeros((N, N))


for _, row in edges_weekends_df.iterrows():

        start_station = row["start_station_id"]
        end_station = row["end_station_id"]
        weight = row["weight"]

        i = station_to_index[start_station]
        j = station_to_index[end_station]


        wij_matrix[i][j] = weight

In [ ]:
wik_sum = wij_matrix.sum(axis=1)
wik_sum = wik_sum.reshape((-1, 1))

In [ ]:
Pij_matrix = wij_matrix / wik_sum
Pij_matrix.sum(axis=1)

In [ ]:
p_t0_matrix = np.zeros((1, N))
p_t0_matrix[0, :] = 1/N
print(p_t0_matrix)

In [ ]:
p_t1_matrix = np.zeros((1, N))

diff = abs(p_t1_matrix - p_t0_matrix). sum()
iteration = 0

while diff > 1e-10:
    p_t1_matrix = p_t0_matrix @ Pij_matrix
    diff = abs(p_t1_matrix - p_t0_matrix). sum()
    p_t0_matrix = p_t1_matrix
    iteration += 1

print(iteration)

stationary_distribution = p_t1_matrix


In [ ]:
sorted_indices = np.argsort(stationary_distribution[0])[::-1]
sorted_probs = stationary_distribution[0][sorted_indices]

top_10_weekend_stations = sorted_indices[:10]
top_10_weekend_probs = sorted_probs[:10]

print("--- Top 10 stations on weekends: ---")


for idx, prob in zip(top_10_weekend_stations, top_10_weekend_probs):

    station_id = index_to_station[idx]

    station_name = stations_weekends_df[stations_weekends_df["station_id"] == station_id]["station_name"].values[0]

    print(station_name)


In [ ]:
# ---------- Top 30 stations ----------

top_30_weekend_indices = sorted_indices[:30]

top_30_weekend_names = []

for idx in top_30_weekend_indices:

    station_id = index_to_station[idx]

    station_name = stations_weekends_df[
        stations_weekends_df["station_id"] == station_id
    ]["station_name"].values[0]

    top_30_weekend_names.append(station_name)


# ---------- Top 10 stations ----------

top_10_weekend_names = []

for idx in top_10_weekend_stations:

    station_id = index_to_station[idx]

    station_name = stations_weekends_df[
        stations_weekends_df["station_id"] == station_id
    ]["station_name"].values[0]

    top_10_weekend_names.append(station_name)


# ---------- Filtering strongest edges ----------

min_weight = 400

filtered_edges_df = edges_weekends_df[
    (edges_weekends_df["start_station_name"] != edges_weekends_df["end_station_name"]) &
    (edges_weekends_df["weight"] >= min_weight)
].copy()


# ---------- Building filtered graph ----------

filtered_graph = nx.from_pandas_edgelist(
    filtered_edges_df,
    source="start_station_name",
    target="end_station_name",
    edge_attr="weight",
    create_using=nx.DiGraph()
)


# ---------- GeoDataFrame ----------

stations_weekends_df["lat"] = stations_weekends_df["lat"].astype(float)
stations_weekends_df["lon"] = stations_weekends_df["lon"].astype(float)

geo_df = gpd.GeoDataFrame(
    stations_weekends_df,
    geometry=gpd.points_from_xy(
        stations_weekends_df["lon"],
        stations_weekends_df["lat"]
    ),
    crs="EPSG:4326"
)

geo_df = geo_df.to_crs(epsg=3857)


# ---------- Positions in EPSG:3857 ----------

positions_3857 = {}

for _, row in geo_df.iterrows():

    positions_3857[row["station_name"]] = (
        row.geometry.x,
        row.geometry.y
    )


# ---------- Stationary distribution dictionary ----------

stationary_dict = {}

for idx, probability in enumerate(stationary_distribution[0]):

    station_id = index_to_station[idx]

    station_name = stations_weekends_df[
        stations_weekends_df["station_id"] == station_id
    ]["station_name"].values[0]

    stationary_dict[station_name] = probability


# ---------- Node sizes proportional to stationary distribution ----------

top_30_node_sizes = [
    300 + 120000 * stationary_dict[node]
    for node in top_30_weekend_names
]

top_10_node_sizes = [
    300 + 120000 * stationary_dict[node]
    for node in top_10_weekend_names
]

remaining_20_weekend_names = [
    node for node in top_30_weekend_names
    if node not in top_10_weekend_names
]

remaining_20_node_sizes = [
    300 + 120000 * stationary_dict[node]
    for node in remaining_20_weekend_names
]


# ---------- Labels ----------

def clean_station_name(name):
    return (
        str(name)
        .strip()
        .replace(" ,", ",")
        .replace(", ", ",")
        .casefold()
    )


group_label_station = clean_station_name("Wellington Arch, Hyde Park")

labels = {}

for node in top_10_weekend_names:

    clean_node = clean_station_name(node)

    # Skip all Hyde Park labels except Wellington Arch
    if "hyde park" in clean_node and clean_node != group_label_station:
        continue

    # Replace Wellington Arch label with grouped Hyde Park label
    if clean_node == group_label_station:
        labels[node] = "5 Hyde Park Stations"
    else:
        labels[node] = node


# ---------- Plot style ----------

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.titlesize"] = 20
plt.rcParams["axes.titleweight"] = "bold"


# ---------- Drawing ----------

fig, ax = plt.subplots(figsize=(16, 12), dpi=250)


# Remaining top 30 stations, faded

nx.draw_networkx_nodes(
    filtered_graph,
    positions_3857,
    nodelist=remaining_20_weekend_names,
    node_size=remaining_20_node_sizes,
    node_color="#FF7A59",
    alpha=0.34,
    edgecolors="#5A2418",
    linewidths=0.25,
    ax=ax
)


# Top 10 stations highlighted

nx.draw_networkx_nodes(
    filtered_graph,
    positions_3857,
    nodelist=top_10_weekend_names,
    node_size=top_10_node_sizes,
    node_color="#FF7A59",
    alpha=1.0,
    edgecolors="#5A2418",
    linewidths=1.0,
    ax=ax
)


# Labels for selected top 10 stations

text_items = nx.draw_networkx_labels(
    filtered_graph,
    positions_3857,
    labels=labels,
    font_size=11,
    font_weight="bold",
    font_color="#2b2b2b",
    font_family="DejaVu Sans",
    ax=ax
)

for _, text in text_items.items():

    text.set_bbox(
        dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.75,
            boxstyle="round,pad=0.25"
        )
    )


# ---------- Map extent ----------

x_values = [
    positions_3857[node][0]
    for node in top_30_weekend_names
]

y_values = [
    positions_3857[node][1]
    for node in top_30_weekend_names
]

ax.set_xlim(
    min(x_values) - padding,
    max(x_values) + padding
)

ax.set_ylim(
    min(y_values) - padding,
    max(y_values) + padding
)


# ---------- Basemap ----------

safe_add_basemap(
    ax,
    source=ctx.providers.CartoDB.Positron
)

# ---------- Hyde Park boundary ----------

hyde_park_boundary = ox.geocode_to_gdf("Hyde Park, London, UK")
hyde_park_boundary = hyde_park_boundary.to_crs(epsg=3857)


hyde_park_boundary.plot(
    ax=ax,
    facecolor="#6A994E",
    edgecolor="none",
    alpha=0.08,
    zorder=1
)

hyde_park_boundary.boundary.plot(
    ax=ax,
    color="#386641",
    linewidth=1.8,
    linestyle="--",
    alpha=0.85,
    zorder=2
)


# ---------- Average route scale bar ----------

add_average_route_scale_bar(
    ax,
    average_weekend_route_km
)


# ---------- Legend ----------

legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        label="10 najważniejszych stacji weekendowych",
        markerfacecolor="#FF7A59",
        markeredgecolor="#5A2418",
        markeredgewidth=0.9,
        markersize=10,
        alpha=1.0
    ),
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        label="Pozostałe 20 stacji",
        markerfacecolor="#FF7A59",
        markeredgecolor="#5A2418",
        markeredgewidth=0.9,
        markersize=10,
        alpha=0.34
    ),
    Line2D(
    [0], [0],
    color="#386641",
    linestyle="--",
    linewidth=1.8,
    label="Granica Hyde Parku"
)
]

legend = ax.legend(
    handles=legend_handles,
    loc="upper left",
    frameon=True,
    facecolor="white",
    edgecolor="#B4BCC2",
    fontsize=12.5,
    borderpad=1.0,
    labelspacing=1.3,
    handletextpad=0.8
)

legend.get_frame().set_alpha(0.9)


# ---------- Title ----------

ax.set_title(
    "Najważniejsze stacje sieci rowerowej Londynu - weekendy",
    color="#2b2b2b",
    pad=18
)

ax.axis("off")
plt.tight_layout()
plt.show()

## Weekdays network

In [ ]:
start_stations_weekdays_df = trip_data[
    [
        "start_station_id",
        "start_station_name",
        "start_lat",
        "start_lon",
    ]
].drop_duplicates()

start_stations_weekdays_df = start_stations_weekdays_df.rename(columns={
    "start_station_id": "station_id",
    "start_station_name": "station_name",
    "start_lat": "lat",
    "start_lon": "lon"
})

start_stations_weekdays_df.head()

In [ ]:
end_stations_weekdays_df = trip_data[
    [
        "end_station_id",
        "end_station_name",
        "end_lat",
        "end_lon",
    ]
].drop_duplicates()

end_stations_weekdays_df = end_stations_weekdays_df.rename(columns={
    "end_station_id": "station_id",
    "end_station_name": "station_name",
    "end_lat": "lat",
    "end_lon": "lon"
})

end_stations_weekdays_df.head()

In [ ]:
stations_weekdays_df = pd.concat([start_stations_weekdays_df, end_stations_weekdays_df], ignore_index=True).drop_duplicates()
print("Number of stations:", len(stations_weekdays_df))

In [ ]:
edges_weekdays_df = trip_data_weekdays.groupby(
    [
        "start_station_id",
        "start_station_name",
        "end_station_id",
        "end_station_name"
    ]
).agg(
    weight=("start_station_id", "count")
).reset_index()

In [ ]:
graph_weekdays = nx.from_pandas_edgelist(
    edges_weekdays_df,
    source="start_station_name",
    target="end_station_name",
    edge_attr="weight",
    create_using=nx.DiGraph()
)
    
positions = {}

stations_weekdays_df["lat"] = stations_weekdays_df["lat"].astype(float)
stations_weekdays_df["lon"] = stations_weekdays_df["lon"].astype(float)

for _, row in stations_weekdays_df.iterrows():

    positions[row["station_name"]] = (
        row["lon"],
        row["lat"]
    )

## Weekdays Markov

In [ ]:
positions = {}

for _, row in stations_weekdays_df.iterrows():

    positions[row["station_name"]] = (
        row["lon"],
        row["lat"]
    )

station_to_index = {
    station_id: idx
    for idx, station_id in enumerate(
        sorted(stations_weekdays_df["station_id"].unique())
    )
}

index_to_station = {
    idx: station_id
    for station_id, idx in station_to_index.items()
}

In [ ]:
N = len(stations_weekdays_df["station_id"].unique())

wij_matrix = np.zeros((N, N))


for _, row in edges_weekdays_df.iterrows():

        start_station = row["start_station_id"]
        end_station = row["end_station_id"]
        weight = row["weight"]

        i = station_to_index[start_station]
        j = station_to_index[end_station]


        wij_matrix[i][j] = weight

In [ ]:
wik_sum = wij_matrix.sum(axis=1)
wik_sum = wik_sum.reshape((-1, 1))

In [ ]:
Pij_matrix = wij_matrix / wik_sum
Pij_matrix.sum(axis=1)

In [ ]:
p_t0_matrix = np.zeros((1, N))
p_t0_matrix[0, :] = 1/N
print(p_t0_matrix)

In [ ]:
p_t1_matrix = np.zeros((1, N))

diff = abs(p_t1_matrix - p_t0_matrix). sum()
iteration = 0

while diff > 1e-10:
    p_t1_matrix = p_t0_matrix @ Pij_matrix
    diff = abs(p_t1_matrix - p_t0_matrix). sum()
    p_t0_matrix = p_t1_matrix
    iteration += 1

print(iteration)

stationary_distribution = p_t1_matrix


In [ ]:
sorted_weekdays_indices = np.argsort(stationary_distribution[0])[::-1]
sorted_weekdays_probs = stationary_distribution[0][sorted_weekdays_indices]

top_10_weekdays_stations = sorted_weekdays_indices[:10]
top_10_weekdays_probs = sorted_weekdays_probs[:10]

print("--- Top 10 stations on weekdays: ---")

for idx, prob in zip(top_10_weekdays_stations, top_10_weekdays_probs):

    station_id = index_to_station[idx]

    station_name = stations_weekdays_df[stations_weekdays_df["station_id"] == station_id]["station_name"].values[0]

    print(station_name)
    

## City of London

In [ ]:
city_of_london_boundary = gpd.read_file(
    "https://mapit.mysociety.org/area/2512.geojson"
)

city_of_london_boundary = city_of_london_boundary.set_crs(
    "EPSG:4326",
    allow_override=True
)

city_of_london_boundary = city_of_london_boundary.to_crs(epsg=3857)

In [ ]:
# ---------- Top 30 stations ----------

top_30_weekdays_indices = sorted_weekdays_indices[:30]

top_30_weekdays_names = []

for idx in top_30_weekdays_indices:

    station_id = index_to_station[idx]

    station_name = stations_weekdays_df[
        stations_weekdays_df["station_id"] == station_id
    ]["station_name"].values[0]

    top_30_weekdays_names.append(station_name)


# ---------- Top 10 stations ----------

top_10_weekdays_names = []

for idx in top_10_weekdays_stations:

    station_id = index_to_station[idx]

    station_name = stations_weekdays_df[
        stations_weekdays_df["station_id"] == station_id
    ]["station_name"].values[0]

    top_10_weekdays_names.append(station_name)


# ---------- Filtering strongest edges ----------

min_weight = 400

filtered_edges_df = edges_weekdays_df[
    (edges_weekdays_df["start_station_name"] != edges_weekdays_df["end_station_name"]) &
    (edges_weekdays_df["weight"] >= min_weight)
].copy()


# ---------- Building filtered graph ----------

filtered_graph = nx.from_pandas_edgelist(
    filtered_edges_df,
    source="start_station_name",
    target="end_station_name",
    edge_attr="weight",
    create_using=nx.DiGraph()
)


# ---------- GeoDataFrame ----------

stations_weekdays_df["lat"] = stations_weekdays_df["lat"].astype(float)
stations_weekdays_df["lon"] = stations_weekdays_df["lon"].astype(float)

geo_df = gpd.GeoDataFrame(
    stations_weekdays_df,
    geometry=gpd.points_from_xy(
        stations_weekdays_df["lon"],
        stations_weekdays_df["lat"]
    ),
    crs="EPSG:4326"
)

geo_df = geo_df.to_crs(epsg=3857)


# ---------- Positions in EPSG:3857 ----------

positions_3857 = {}

for _, row in geo_df.iterrows():

    positions_3857[row["station_name"]] = (
        row.geometry.x,
        row.geometry.y
    )


# ---------- Stationary distribution dictionary ----------

stationary_dict = {}

for idx, probability in enumerate(stationary_distribution[0]):

    station_id = index_to_station[idx]

    station_name = stations_weekdays_df[
        stations_weekdays_df["station_id"] == station_id
    ]["station_name"].values[0]

    stationary_dict[station_name] = probability


# ---------- Node sizes proportional to stationary distribution ----------

top_30_node_sizes = [
    300 + 120000 * stationary_dict[node]
    for node in top_30_weekdays_names
]

top_10_node_sizes = [
    300 + 120000 * stationary_dict[node]
    for node in top_10_weekdays_names
]

remaining_20_weekdays_names = [
    node for node in top_30_weekdays_names
    if node not in top_10_weekdays_names
]

remaining_20_node_sizes = [
    300 + 120000 * stationary_dict[node]
    for node in remaining_20_weekdays_names
]


# ---------- Labels ----------

labels_to_skip = [
    "Waterloo Station 1, Waterloo",
    "Queen Street 1, Bank",
]

labels = {}

for node in top_10_weekdays_names:

    if node in labels_to_skip:
        continue

    elif node == "Waterloo Station 3, Waterloo":
        labels[node] = "2 Waterloo Stations"

    elif node == "Queen Street 2, Bank":
        labels[node] = "2 Queen Street Stations"

    else:
        labels[node] = node


# ---------- Plot style ----------

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.titlesize"] = 20
plt.rcParams["axes.titleweight"] = "bold"


# ---------- Drawing ----------

fig, ax = plt.subplots(figsize=(16, 12), dpi=250)


# Remaining top 30 stations, faded

nx.draw_networkx_nodes(
    filtered_graph,
    positions_3857,
    nodelist=remaining_20_weekdays_names,
    node_size=remaining_20_node_sizes,
    node_color="#2EC4B6",
    alpha=0.34,
    edgecolors="#123C3A",
    linewidths=0.25,
    label="Pozostałe 20 ważnychstacji",
    ax=ax
)


# Top 10 stations highlighted

nx.draw_networkx_nodes(
    filtered_graph,
    positions_3857,
    nodelist=top_10_weekdays_names,
    node_size=top_10_node_sizes,
    node_color="#2EC4B6",
    alpha=1.0,
    edgecolors="#123C3A",
    linewidths=1.0,
    label="10 najważniejszych stacji dni roboczych",
    ax=ax
)


# Labels for selected top 10 stations

text_items = nx.draw_networkx_labels(
    filtered_graph,
    positions_3857,
    labels=labels,
    font_size=11,
    font_weight="bold",
    font_color="#2b2b2b",
    font_family="DejaVu Sans",
    ax=ax
)

for _, text in text_items.items():

    text.set_bbox(
        dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.75,
            boxstyle="round,pad=0.25"
        )
    )


# ---------- Map extent ----------

x_values = [
    positions_3857[node][0]
    for node in top_30_weekdays_names
]

y_values = [
    positions_3857[node][1]
    for node in top_30_weekdays_names
]

ax.set_xlim(
    min(x_values) - padding,
    max(x_values) + padding
)

ax.set_ylim(
    min(y_values) - padding,
    max(y_values) + padding
)


# ---------- Basemap ----------

safe_add_basemap(
    ax,
    source=ctx.providers.CartoDB.Positron
)

# ---------- City of London boundary ----------

city_of_london_boundary.plot(
    ax=ax,
    facecolor="#2EC4B6",
    edgecolor="none",
    alpha=0.06,
    zorder=1
)

city_of_london_boundary.boundary.plot(
    ax=ax,
    color="#4F7C78",
    linewidth=1.8,
    linestyle="--",
    alpha=0.9,
    zorder=2
)


# ---------- Average route scale bar ----------

add_average_route_scale_bar(
    ax,
    average_weekday_route_km
)


# ---------- Legend and title ----------

legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        label="10 najważniejszych stacji dni roboczych",
        markerfacecolor="#2EC4B6",
        markeredgecolor="#123C3A",
        markeredgewidth=0.9,
        markersize=10,
        alpha=1.0
    ),
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        label="Pozostałe 20 ważnych stacji",
        markerfacecolor="#2EC4B6",
        markeredgecolor="#123C3A",
        markeredgewidth=0.9,
        markersize=10,
        alpha=0.34
    ),
    Line2D(
    [0], [0],
    color="#4F7C78",
    linestyle="--",
    linewidth=1.8,
    label="Granica City of London"
    )   
]

legend = ax.legend(
    handles=legend_handles,
    loc="upper left",
    frameon=True,
    facecolor="white",
    edgecolor="#B4BCC2",
    fontsize=12.5,
    borderpad=1.0,
    labelspacing=1.3,
    handletextpad=0.8
)

legend.get_frame().set_alpha(0.9)


# ---------- Title ----------

ax.set_title(
    "Najważniejsze stacje sieci rowerowej Londynu - dni robocze",
    color="#2b2b2b",
    pad=18
)

ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ---------- Weekend vs weekday top stations on one map ----------
# Delicate weekday/weekend background + highlighted shared stations


# ---------- Sets and overlap ----------

weekend_top_30_set = set(top_30_weekend_names)
weekday_top_30_set = set(top_30_weekdays_names)

overlap_names = [
    name for name in top_30_weekend_names
    if name in weekday_top_30_set
]


# ---------- Top 5 shared stations based on combined rank ----------

weekend_rank = {
    station_name: rank
    for rank, station_name in enumerate(top_30_weekend_names, start=1)
}

weekday_rank = {
    station_name: rank
    for rank, station_name in enumerate(top_30_weekdays_names, start=1)
}

top_5_shared_names = sorted(
    overlap_names,
    key=lambda name: (
        weekend_rank[name] + weekday_rank[name],
        weekend_rank[name],
        weekday_rank[name]
    )
)[:5]


# ---------- Shared station positions ----------

comparison_stations_df = pd.concat(
    [stations_weekends_df, stations_weekdays_df],
    ignore_index=True
).drop_duplicates(subset=["station_name"]).copy()

comparison_stations_df["lat"] = comparison_stations_df["lat"].astype(float)
comparison_stations_df["lon"] = comparison_stations_df["lon"].astype(float)

comparison_geo_df = gpd.GeoDataFrame(
    comparison_stations_df,
    geometry=gpd.points_from_xy(
        comparison_stations_df["lon"],
        comparison_stations_df["lat"]
    ),
    crs="EPSG:4326"
)

comparison_geo_df = comparison_geo_df.to_crs(epsg=3857)

comparison_positions_3857 = {
    row["station_name"]: (row.geometry.x, row.geometry.y)
    for _, row in comparison_geo_df.iterrows()
}


# ---------- Labels only for top 5 shared stations ----------

def clean_station_name(name):
    return (
        str(name)
        .strip()
        .replace(" ,", ",")
        .replace(", ", ",")
        .casefold()
    )


labels_to_skip = [
    "Waterloo Station 1, Waterloo",
    "Albert Gate, Hyde Park"
]

comparison_labels = {}

for node in top_5_shared_names:

    clean_node = clean_station_name(node)

    if clean_node in [clean_station_name(name) for name in labels_to_skip]:
        continue

    if clean_node == clean_station_name("Waterloo Station 3, Waterloo"):
        comparison_labels[node] = "2 Waterloo Stations"

    elif clean_node == clean_station_name("Hyde Park Corner, Hyde Park"):
        comparison_labels[node] = "2 Hyde Park Corner Stations"

    else:
        comparison_labels[node] = node


# ---------- Plot style ----------

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.titlesize"] = 20
plt.rcParams["axes.titleweight"] = "bold"


# ---------- Helper function ----------

def scatter_station_group(
    ax,
    station_names,
    color,
    edge_color,
    label,
    size,
    alpha,
    zorder,
    linewidths=0.8
):
    plotted_names = [
        station_name
        for station_name in station_names
        if station_name in comparison_positions_3857
    ]

    if not plotted_names:
        return

    x_values = [
        comparison_positions_3857[name][0]
        for name in plotted_names
    ]

    y_values = [
        comparison_positions_3857[name][1]
        for name in plotted_names
    ]

    ax.scatter(
        x_values,
        y_values,
        s=size,
        c=color,
        alpha=alpha,
        edgecolors=edge_color,
        linewidths=linewidths,
        label=label,
        zorder=zorder
    )


# ---------- Drawing ----------

fig, ax = plt.subplots(figsize=(16, 12), dpi=250)


# ---------- Map extent ----------

comparison_top_30_names = list(dict.fromkeys(
    top_30_weekend_names + top_30_weekdays_names
))

x_values = [
    comparison_positions_3857[node][0]
    for node in comparison_top_30_names
    if node in comparison_positions_3857
]

y_values = [
    comparison_positions_3857[node][1]
    for node in comparison_top_30_names
    if node in comparison_positions_3857
]

ax.set_xlim(
    min(x_values) - padding,
    max(x_values) + padding
)

ax.set_ylim(
    min(y_values) - padding,
    max(y_values) + padding
)


# ---------- Basemap ----------

safe_add_basemap(
    ax,
    source=ctx.providers.CartoDB.Positron
)


# ---------- City of London boundary ----------

city_of_london_boundary.plot(
    ax=ax,
    facecolor="#2EC4B6",
    edgecolor="none",
    alpha=0.05,
    zorder=1
)

city_of_london_boundary.boundary.plot(
    ax=ax,
    color="#4F7C78",
    linewidth=1.7,
    linestyle="--",
    alpha=0.85,
    zorder=2
)


# ---------- Hyde Park boundary ----------

hyde_park_boundary = ox.geocode_to_gdf("Hyde Park, London, UK")
hyde_park_boundary = hyde_park_boundary.to_crs(epsg=3857)

hyde_park_boundary.plot(
    ax=ax,
    facecolor="#6A994E",
    edgecolor="none",
    alpha=0.08,
    zorder=1
)

hyde_park_boundary.boundary.plot(
    ax=ax,
    color="#386641",
    linewidth=1.7,
    linestyle="--",
    alpha=0.85,
    zorder=2
)


# ---------- Weekday top 30 stations - delicate background ----------

scatter_station_group(
    ax=ax,
    station_names=top_30_weekdays_names,
    color="#2EC4B6",
    edge_color="#123C3A",
    label="30 najważniejszych stacji dni roboczych",
    size=260,
    alpha=0.28,
    zorder=3,
    linewidths=0.5
)


# ---------- Weekend top 30 stations - delicate background ----------

scatter_station_group(
    ax=ax,
    station_names=top_30_weekend_names,
    color="#FF7A59",
    edge_color="#5A2418",
    label="30 najważniejszych stacji weekendowych",
    size=260,
    alpha=0.28,
    zorder=4,
    linewidths=0.5
)


# ---------- Shared stations - strong highlight ----------

scatter_station_group(
    ax=ax,
    station_names=overlap_names,
    color="#F2C94C",
    edge_color="#4A3B00",
    label="Stacje wspólne dla obu okresów",
    size=430,
    alpha=0.95,
    zorder=6,
    linewidths=1.1
)


# ---------- Top 5 shared stations - slightly larger ----------

scatter_station_group(
    ax=ax,
    station_names=top_5_shared_names,
    color="#D4A017",
    edge_color="#4A3B00",
    label="5 najważniejszych wspólnych stacji",
    size=570,
    alpha=1.0,
    zorder=7,
    linewidths=1.3
)


# ---------- Labels for top 5 shared stations ----------

for node, label in comparison_labels.items():

    if node not in comparison_positions_3857:
        continue

    x, y = comparison_positions_3857[node]

    ax.text(
        x,
        y,
        label,
        fontsize=10.5,
        fontweight="bold",
        color="#2b2b2b",
        ha="center",
        va="center",
        bbox=dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.78,
            boxstyle="round,pad=0.25"
        ),
        zorder=8
    )


# ---------- Legend ----------

legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        label="5 najważniejszych wspólnych stacji",
        markerfacecolor="#D4A017",
        markeredgecolor="#4A3B00",
        markeredgewidth=0.9,
        markersize=10,
        alpha=1.0
    ),
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        label="Stacje wspólne dla obu okresów",
        markerfacecolor="#F2C94C",
        markeredgecolor="#4A3B00",
        markeredgewidth=0.9,
        markersize=10,
        alpha=0.95
    ),
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        label="30 najważniejszych stacji weekendowych",
        markerfacecolor="#FF7A59",
        markeredgecolor="#5A2418",
        markeredgewidth=0.9,
        markersize=10,
        alpha=0.28
    ),
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        label="30 najważniejszych stacji dni roboczych",
        markerfacecolor="#2EC4B6",
        markeredgecolor="#123C3A",
        markeredgewidth=0.9,
        markersize=10,
        alpha=0.28
    ),
    Line2D(
        [0], [0],
        color="#4F7C78",
        linestyle="--",
        linewidth=1.7,
        label="Granica City of London"
    ),
    Line2D(
        [0], [0],
        color="#386641",
        linestyle="--",
        linewidth=1.7,
        label="Granica Hyde Parku"
    )
]

legend = ax.legend(
    handles=legend_handles,
    loc="upper left",
    frameon=True,
    facecolor="white",
    edgecolor="#B4BCC2",
    fontsize=12.5,
    borderpad=1.0,
    labelspacing=1.3,
    handletextpad=0.8
)

legend.get_frame().set_alpha(0.9)


# ---------- Title ----------

ax.set_title(
    "Porównanie najważniejszych stacji - dni robocze i weekendy",
    color="#2b2b2b",
    pad=18
)

ax.axis("off")
plt.tight_layout()
plt.show()

# ---------- Duration distributions and histograms ----------


In [ ]:
# ---------- Preparing data ----------

duration_distribution_data = pd.concat(
    [
        trip_data_weekdays.assign(day_type="Dni robocze"),
        trip_data_weekends.assign(day_type="Weekendy")
    ],
    ignore_index=True
)

duration_distribution_data["duration_min"] = duration_distribution_data[
    "total_duration_minutes"
]

duration_distribution_data = duration_distribution_data.dropna(
    subset=["duration_min"]
)


# ---------- Duration ranges ----------

duration_bins = [0, 5, 10, 15, 20, 30, 45, 60, 90, 120, np.inf]

duration_labels = [
    "0-5", "5-10", "10-15", "15-20", "20-30",
    "30-45", "45-60", "60-90", "90-120", "120+"
]

duration_distribution_data["duration_range"] = pd.cut(
    duration_distribution_data["duration_min"],
    bins=duration_bins,
    labels=duration_labels,
    right=False
)


# ---------- Duration distribution table ----------

duration_distribution = (
    duration_distribution_data
    .groupby(["day_type", "duration_range"], observed=False)
    .size()
    .rename("liczba_przejazdow")
    .reset_index()
)

duration_distribution["udzial_%"] = (
    duration_distribution["liczba_przejazdow"]
    / duration_distribution.groupby("day_type")["liczba_przejazdow"].transform("sum")
    * 100
)

duration_distribution_pivot = duration_distribution.pivot(
    index="duration_range",
    columns="day_type",
    values=["liczba_przejazdow", "udzial_%"]
)

display(duration_distribution_pivot.round(2))


In [ ]:
# ---------- Histograms ----------

duration_histogram_data = duration_distribution_data[
    duration_distribution_data["duration_min"].between(0, 120)
].copy()

colors = {
    "Dni robocze": "#1F77B4",   # niebieski
    "Weekendy": "#FF7F0E"       # pomarańczowy
}

bins = np.arange(0, 125, 5)


# ---------- Linear scale ----------

plt.figure(figsize=(13, 7))

sns.histplot(
    data=duration_histogram_data,
    x="duration_min",
    hue="day_type",
    bins=bins,
    stat="percent",
    common_norm=False,
    element="bars",
    alpha=0.55,
    palette=colors,
    edgecolor="white"
)

ax = plt.gca()
ax.legend_.set_title("")


plt.title("Rozkład czasu przejazdów w dni robocze i weekendy")
plt.xlabel("Czas przejazdu [min]")
plt.ylabel("Udział przejazdów [%]")
plt.xlim(left=0)
plt.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:
# ---------- Stationary probability comparison ----------

top_n = min(
    len(top_10_weekdays_names),
    len(top_10_weekdays_probs),
    len(top_10_weekend_names),
    len(top_10_weekend_probs)
)

pi_station_comparison = pd.DataFrame(
    {
        "pi_dni_robocze": top_10_weekdays_probs[:top_n],
        "stacja_dni_robocze": top_10_weekdays_names[:top_n],
        "pi_weekendy": top_10_weekend_probs[:top_n],
        "stacja_weekendy": top_10_weekend_names[:top_n],
    },
    index=range(1, top_n + 1)
)

pi_station_comparison.index.name = "miejsce"

pi_station_comparison_csv_path = DATA_DIR / "pi_station_comparison_weekdays_weekends.csv"

pi_station_comparison.to_csv(
    pi_station_comparison_csv_path,
    encoding="utf-8",
    float_format="%.8f"
)

pi_station_comparison_display = pi_station_comparison.copy()

pi_station_comparison_display["pi_dni_robocze"] = (
    pi_station_comparison_display["pi_dni_robocze"].map("{:.8f}".format)
)

pi_station_comparison_display["pi_weekendy"] = (
    pi_station_comparison_display["pi_weekendy"].map("{:.8f}".format)
)

display(pi_station_comparison_display)
print(f"Zapisano CSV: {pi_station_comparison_csv_path}")
